In [5]:
# !pip install --upgrade openai pydantic

README.md		gm_instruction_generator.py  true_seed
__pycache__		gm_json_generator.py	     vm_logic_validator.py
ds_main.ipynb		gm_sample_picker.py	     vm_supervisor.py
ds_system_prompts.py	growing_seed		     vm_syntax_validator.py
eval_dataset		img
generated_instructions	logs


In [ ]:
import os
import json
import random
import logging
from typing import List, Tuple, Any, Optional

from gm_instruction_generator import GM_InstructionGenerator
from gm_json_generator import GM_JSONGenerator
from vm_logic_validator import VM_LogicValidator
from vm_syntax_validator import VM_SyntaxValidator
from vm_supervisor import VM_Supervisor
from gm_sample_picker import SamplePicker
from ds_system_prompts import SystemPromptRepository

class DataSpinner:
    def __init__(
        self,
        seed_folder: str,
        log_folder: str = "logs"
    ):
        """
        Initialize DataSpinner with seed and log folders. Modules and prompts are configured here.
        """
        with open("openai_key.txt") as f:
            os.environ["OPENAI_API_KEY"] = f.read().strip()
        
        # ensure directories exist
        self.seed_folder = seed_folder
        os.makedirs(seed_folder, exist_ok=True)
        self.log_folder = log_folder
        os.makedirs(log_folder, exist_ok=True)

        # setup logger
        self.logger = logging.getLogger("DataSpinner")
        self.logger.setLevel(logging.INFO)
        log_path = os.path.join(log_folder, "dataspinner.log")
        fh = logging.FileHandler(log_path)
        fh.setLevel(logging.INFO)
        formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
        fh.setFormatter(formatter)
        if not self.logger.handlers:
            self.logger.addHandler(fh)

        # initialize modules
        self.picker = SamplePicker(seed_folder=seed_folder)
        self.instruction_gen = GM_InstructionGenerator(None,"gpt-4.1")
        self.json_gen = GM_JSONGenerator(None,"gpt-4.1")
        self.logic_val = VM_LogicValidator(None,"gpt-4.1-mini")
        self.syntax_val = VM_SyntaxValidator(None,"gpt-4.1-mini")
        self.supervisor = VM_Supervisor(None,"gpt-4.1-mini")

        # load system prompts
        self.prompt_instr = SystemPromptRepository.get_instruction_generation_prompt()
        self.prompt_hard_instr = SystemPromptRepository.get_hard_instruction_generation_prompt()
        self.prompt_json = SystemPromptRepository.get_json_generation_prompt()
        self.prompt_logic = SystemPromptRepository.get_logic_validation_prompt()
        self.prompt_syntax = SystemPromptRepository.get_syntax_validation_prompt()
        self.prompt_sup = SystemPromptRepository.get_supervisor_prompt()

    def run_once(
        self,
        seed_type: str,
        batch_size: int,
        num_examples: int
    ) -> int:
        """
        Execute one iteration of the pipeline for a given seed_type:
          1) sample instruction examples
          2) generate new instructions
          3) convert new instructions to JSON
          4) logic & syntax validation on new JSON
          5) supervise and keep only valid JSON
          6) append valid instruction/JSON pairs to seed dataset
        Returns the number of new pairs appended.
        """
        # 1) sample seed examples
        self.logger.info(f"[Progress] Step 1/6: Sampling examples for '{seed_type}' ({num_examples} needed)")
        data_sample = self.picker.get_full_examples(seed_type, num_examples)
        if not data_sample:
            self.logger.warning(f"Not enough samples for type '{seed_type}'")
            self.picker.reset(seed_type)
            self.logger.warning(f"Reset seed dataset of type:'{seed_type}'")
            data_sample = self.picker.get_full_examples(seed_type, num_examples)

        sample_instructions, sample_jsons = zip(*[
            (ex["instruction"], ex["json"]) for ex in data_sample["examples"]
        ])
        self.logger.info(f"[Progress] Step 1/6 complete: sampled {len(sample_instructions)} examples")

        # 2) generate new instructions (for HARD type, use dedicated HARD prompt)
        self.logger.info(f"[Progress] Step 2/6: Generating {batch_size} new instructions")
        if seed_type == "hard":
            new_instructions, _ = self.instruction_gen.generate_instructions(
            list(sample_instructions), seed_type, self.prompt_hard_instr, batch_size)
        else:
            new_instructions, _ = self.instruction_gen.generate_instructions(
                list(sample_instructions), seed_type, self.prompt_instr, batch_size)
        self.logger.info(f"[Progress] Step 2/6 complete: generated {len(new_instructions)} instructions")

        # 3) convert to JSON
        self.logger.info("[Progress] Step 3/6: Converting instructions to JSON")
        instr_json_pairs, _ = self.json_gen.generate_json(
            new_instructions, data_sample, seed_type, self.prompt_json
        )
        self.logger.info(f"[Progress] Step 3/6 complete: converted to {len(instr_json_pairs)} JSON pairs")

        # 4) logic validation
        self.logger.info("[Progress] Step 4/6: Performing logic validation")
        logic_ok, _ = self.logic_val.validate_logic(
            instr_json_pairs, seed_type, self.prompt_logic
        )
        self.logger.info(f"[Progress] Step 4/6 complete: {len(logic_ok)} passed logic check")

        # 5) syntax validation
        self.logger.info("[Progress] Step 5/6: Performing syntax validation")
        syntax_ok, _ = self.syntax_val.validate_syntax(
            logic_ok, seed_type, self.prompt_syntax
        )
        self.logger.info(f"[Progress] Step 5/6 complete: {len(syntax_ok)} passed syntax check")

        ''' Get rid of final supervisor as it detected too many false negatives in many cases
            Fine-tuning it's individual prompt may resolve this issue, but at this point this component just seems to be irrelevant
        # 6) supervision and collect invalid
        self.logger.info("[Progress] Step 6/6: Performing supervision")
        final_ok, invalid_pairs, _ = self.supervisor.supervise(
            syntax_ok, seed_type, self.prompt_sup
        )
        self.logger.info(f"[Progress] Step 6/6 complete: {len(final_ok)} valid pairs, {len(invalid_pairs)} rejected")
        # log all rejection reasons
        for inst, reason in invalid_pairs:
            self.logger.warning(f"Rejected pair for instruction '{inst}': {reason}")
        '''
        
        # prepare new entries
        new_entries = [
            {"instruction": instr, "json": js if isinstance(js, dict) else js.dict()}
            for instr, js in syntax_ok #instead of final_ok 
        ]
        if new_entries:
            self.picker.append_new_pairs(seed_type, new_entries)
            self.logger.info(f"Appended {len(new_entries)} new pairs to seed_{seed_type}.json")
        else:
            self.logger.info("No valid pairs to append")

        return len(new_entries)

    def run_loop(
        self,
        seed_type: str,
        iterations: int,
        batch_size: int,
        num_examples: int
    ) -> None:
        """
        Run multiple iterations of the pipeline with given parameters.
        Logs progress for each iteration.
        """
        total_added = 0
        for i in range(iterations):
            percent = (i + 1) / iterations * 100
            self.logger.info(f"=== Iteration {i+1}/{iterations} ({percent:.1f}%) ===")
            try:
                added = self.run_once(seed_type, batch_size, num_examples)
                total_added += added
            except Exception as e:
                spinner.logger.error(f"An error occurred while calling the API - run_once(): {e}", exc_info=True)
                added = None
        self.logger.info(f"Finished run_loop: {total_added} total new pairs added.")



# 1. Instantiate once (point it at your seed folder)
spinner = DataSpinner(seed_folder="growing_seed")

# 2. Run the pipeline in a loop:
#    seed_type:    which difficulty to pull from (“easy”, “medium”, “hard”)
#    iterations:   how many times to run the full pipeline
#    batch_size:   how many new instructions to generate each iteration
#    num_examples: how many seed examples to sample each iteration
for seed_type, iterations in [
    ("easy",   0),
    ("medium", 0),
    ("hard",   1500),
]:
    try:
        spinner.logger.info(f"Starting run_loop for '{seed_type}' with {iterations} iterations")
        spinner.run_loop(
            seed_type=seed_type,
            iterations=iterations,
            batch_size=1,
            num_examples=2
        )
    except Exception as e:
        # Log full traceback
        spinner.logger.error(f"Exception during run_loop('{seed_type}'):", exc_info=True)
